# Final Cleanup and Submission Preparation

This notebook section only checks and prepares files for final submission.


## Reproducibility
Seeds are used so training and splitting results can be repeated.
The train/validation split is stratified so each class keeps a similar ratio in both sets.
The best validation model is saved so we keep the strongest checkpoint for evaluation and app inference.


In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch


In [ ]:
# Project root paths
ROOT = Path(".")
PLOTS_DIR = ROOT / "assets" / "plots"
WEIGHTS_DIR = ROOT / "assets" / "weights"
APP_DIR = ROOT / "app"

# Create required folders if they are missing
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

# Folder purpose comments:
# - assets/plots: stores generated EDA/evaluation figures
# - assets/weights: stores trained model weights
# - app: stores Shiny web application files


In [ ]:
# Reproducibility helper

def set_seed(seed: int = 42) -> None:
    """Set Python, NumPy, and PyTorch seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
# Submission CSV validation helper
ALLOWED_CLASSES = {
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
}


def validate_submission_file(submission_path: str = "submission.csv") -> pd.DataFrame:
    """Load and validate submission.csv format and class values."""
    path = Path(submission_path)
    if not path.exists():
        raise FileNotFoundError(f"Missing submission file: {path}")

    # File has no header by challenge requirement
    sub_df = pd.read_csv(path, header=None)

    # Must contain exactly two columns
    if sub_df.shape[1] != 2:
        raise ValueError("submission.csv must contain exactly 2 columns.")

    sub_df.columns = ["file_name", "predicted_class"]

    # No missing predictions allowed
    if sub_df["predicted_class"].isna().any():
        raise ValueError("submission.csv has missing predicted classes.")

    # All classes must be from allowed set
    invalid = sorted(set(sub_df["predicted_class"].unique()) - ALLOWED_CLASSES)
    if invalid:
        raise ValueError(f"submission.csv has invalid classes: {invalid}")

    print("Submission preview (first 10 rows):")
    print(sub_df.head(10))
    print(f"Total predictions: {len(sub_df)}")
    print("Unique predicted classes:", sorted(sub_df["predicted_class"].unique()))

    return sub_df


In [ ]:
def final_project_check() -> None:
    """Print final file/folder checklist for submission readiness."""

    checks = [
        ("k12443705.ipynb", ROOT / "k12443705.ipynb"),
        ("app/app.py", ROOT / "app" / "app.py"),
        ("assets/weights/best_model.pt", ROOT / "assets" / "weights" / "best_model.pt"),
        ("assets/plots/random_samples.png", ROOT / "assets" / "plots" / "random_samples.png"),
        ("assets/plots/average_pixel_distribution.png", ROOT / "assets" / "plots" / "average_pixel_distribution.png"),
        ("assets/plots/average_brightness.png", ROOT / "assets" / "plots" / "average_brightness.png"),
        ("assets/plots/training_history.png", ROOT / "assets" / "plots" / "training_history.png"),
        ("assets/plots/confusion_matrix.png", ROOT / "assets" / "plots" / "confusion_matrix.png"),
        ("assets/plots/misclassified_samples.png", ROOT / "assets" / "plots" / "misclassified_samples.png"),
        ("submission.csv", ROOT / "submission.csv"),
        ("README.md", ROOT / "README.md"),
    ]

    for label, path in checks:
        if path.exists():
            print(f"[OK] {label} exists")
        else:
            print(f"[MISSING] {label}")


In [ ]:
# Optional run block for final checks
# set_seed(42)
# _ = validate_submission_file("submission.csv")
# final_project_check()
